# Teti et al. (2019) Replication — Twitter Sentiment & Stock Prices
**Ashley Thompson — Capstone**

Replication of: Teti, Dallocchio & Aniasi (2019), *The relationship between twitter and stock prices. Evidence from the US technology industry*, **Technological Forecasting & Social Change** 149, 119747.

### What this notebook does
- **SET 1** — Pooled OLS: Twitter sentiment, polarity-broken tweet counts, news sentiment, and combinations, predicting percentage daily return (HC3-robust SEs)
- **SET 2** — Panel FE with binary treatment: interaction of lagged sentiment × high/low social-media coverage dummy at lags 0–4
- **SET 3** — Same as SET 2 using a polarity-weighted sentiment index (ts_b) that strips neutral-tweet dilution from the Bloomberg composite
- **Baxter-King filter** applied to price series within each ticker before constructing filtered returns

> **Before running:** Runtime → Change runtime type → T4 GPU (optional, not required for this notebook)

## 1. Install dependencies

In [ ]:
!pip install -q statsmodels linearmodels openpyxl

## 2. Load data
**Option A — Google Drive (recommended)**  
Upload `panel_long_extended.csv` (if you have it) or `panel_long.csv` to Drive first.

**Option B — Manual upload**  
Run the upload cell and select the CSV from your computer.

Run one option, then skip the other.

In [ ]:
# Option A — Google Drive
from google.colab import drive
drive.mount('/content/drive')
# Use extended file if available, otherwise fall back to base panel
import os
DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/Capstone/'
DATA_PATH = DRIVE_DIR + ('panel_long_extended.csv' if os.path.exists(DRIVE_DIR + 'panel_long_extended.csv') else 'panel_long.csv')
print(f'Using: {DATA_PATH}')

In [ ]:
# GitHub output setup — run after data is loaded
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git', REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

## 3. Imports and data preparation

In [ ]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from statsmodels.tsa.filters.bk_filter import bkfilter

warnings.filterwarnings('ignore')
np.random.seed(42)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
long = long.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')
long.head()

In [ ]:
# Check which extended fields are available.
# twitter_neu_count is imputed (not a Bloomberg field) — see next cell.
# Twitter followers are no longer available from Bloomberg; the group dummy
# uses a market-cap median split as the permanent structural proxy.
HAS_POS_COUNT  = 'twitter_pos_count' in long.columns
HAS_MARKET_RET = 'market_return' in long.columns

print(f'twitter_pos_count (Bloomberg pull):  {"AVAILABLE" if HAS_POS_COUNT else "MISSING — SET 3 will be approximated"}')
print(f'twitter_neu_count:                   will be imputed (total - pos - neg)')
print(f'Market index return control:         {"AVAILABLE" if HAS_MARKET_RET else "MISSING — omitted from SET 1 controls"}')
print(f'Twitter followers:                   unavailable from Bloomberg — group dummy uses mkt_cap split')

In [ ]:
# ── Impute twitter_neu_count ──────────────────────────────────────────────
# Bloomberg provides total tweet count (twitter_count) and polarity-broken
# counts for positive (twitter_pos_count) and negative (twitter_neg_count).
# Neutral count is not a published Bloomberg field, but it is the arithmetic
# residual: neu = total - pos - neg.
#
# Constraints enforced:
#   1. Result must be >= 0. Negative values indicate Bloomberg rounding or
#      classification edge cases; clip to 0 rather than propagate bad data.
#   2. Imputation requires all three inputs to be non-missing. Rows where
#      any component is NaN remain NaN — they are dropped later in the
#      complete-case subsets used by SET 1 Model 2 and SET 3.

if HAS_POS_COUNT and 'twitter_neg_count' in long.columns and 'twitter_count' in long.columns:
    long['twitter_neu_count'] = (
        long['twitter_count'] - long['twitter_pos_count'] - long['twitter_neg_count']
    ).clip(lower=0)

    n_imputed  = long['twitter_neu_count'].notna().sum()
    n_negative = (
        (long['twitter_count'] - long['twitter_pos_count'] - long['twitter_neg_count']) < 0
    ).sum()

    print(f'twitter_neu_count imputed:  {n_imputed:,} obs')
    print(f'  Rows clipped to 0 (rounding residuals): {n_negative:,}')
    print(f'  Mean neutral count:  {long["twitter_neu_count"].mean():.1f}')
    print(f'  Max  neutral count:  {long["twitter_neu_count"].max():.0f}')

    HAS_POS_NEU = True
else:
    if not HAS_POS_COUNT:
        print('[SKIPPED] twitter_pos_count not found — cannot impute twitter_neu_count.')
        print('  Pull TWITTER_POS_SENTIMENT_COUNT from Bloomberg and re-run.')
    HAS_POS_NEU = False

In [ ]:
# Percentage return (Teti's dependent variable)
long['pct_return'] = (long['px_close'] - long['px_open']) / long['px_open'] * 100
long['volume_mil'] = long['volume'] / 1e6

# Baxter-King bandpass filter applied to closing price within each ticker
# Isolates cyclical component (removes trend + high-freq noise)
# Parameters: cycles between 5 and 90 trading days, K=12
BK_LOW, BK_HIGH, BK_K = 5, 90, 12

def apply_bk_to_price(series):
    arr = series.values.astype(float)
    mask = ~np.isnan(arr)
    if mask.sum() < 2 * BK_K + 1:
        return pd.Series(np.nan, index=series.index)
    filtered = np.full_like(arr, np.nan)
    filtered_vals = bkfilter(arr[mask], low=BK_LOW, high=BK_HIGH, K=BK_K)
    filtered[np.where(mask)[0]] = filtered_vals
    diff = np.diff(filtered, prepend=np.nan)
    return pd.Series(diff, index=series.index)

long['bk_return'] = long.groupby('ticker')['px_close'].transform(apply_bk_to_price)

# Sentiment lags 0-4 (for SET 2 and SET 3)
for k in range(5):
    long[f'ts_lag_{k}'] = long.groupby('ticker')['twitter_sent'].shift(k)

# Polarity-weighted ts_b index (Teti SET 3)
if HAS_POS_NEU:
    denom = long['twitter_pos_count'] + long['twitter_neu_count'] + long['twitter_neg_count']
    long['ts_b'] = (long['twitter_pos_count'] - long['twitter_neg_count']) / denom.replace(0, np.nan)
    ts_b_note = 'ts_b = (pos - neg) / (pos + neg + neu)'
else:
    long['ts_b'] = long['twitter_sent'].apply(lambda x: np.sign(x) * x**2 if pd.notna(x) else np.nan)
    ts_b_note = 'ts_b ≈ sign(s)·s² [APPROX — pull TWITTER_POS_SENTIMENT_COUNT for exact formula]'

for k in range(5):
    long[f'tsb_lag_{k}'] = long.groupby('ticker')['ts_b'].shift(k)

# ------ Group dummy: high vs. low social-media coverage -------------------
# Bloomberg no longer provides TWITTER_FOLLOWERS. Group dummy is permanently
# defined via a market-cap median split across tickers. Large-cap firms have
# materially greater social media reach and constitute the high-coverage group.
ticker_med_cap = long.groupby('ticker')['mkt_cap'].median()
high_cap_tickers = ticker_med_cap[ticker_med_cap >= ticker_med_cap.median()].index
long['group'] = long['ticker'].isin(high_cap_tickers).astype(int)
group_note = 'group=1 if ticker median mkt_cap >= cross-sectional median (market-cap proxy for social-media reach)'

print(f'Variables constructed.')
print(f'Group dummy: {group_note}')
print(f'  High-coverage tickers: {long["group"].eq(1).groupby(long["ticker"]).first().sum()}')
print(f'ts_b index: {ts_b_note}')
print(f'BK-filtered returns: {long["bk_return"].notna().sum():,} non-missing obs')

## 4. SET 1 — Pooled OLS
Replicates Table 3 of Teti et al. Four specifications:
1. Twitter sentiment score (Bloomberg composite)
2. Polarity-broken tweet counts (pos, neu, neg) — **requires extended Bloomberg data**
3. News sentiment
4. Twitter + news together

Controls: trading volume (mil shares) + market index return (if available).  
SE: HC3 heteroskedasticity-robust.

In [ ]:
def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.1:  return '*'
    return ''

base_ctrl = ' + volume_mil' + (' + market_return' if HAS_MARKET_RET else '')

m1 = smf.ols(f'pct_return ~ twitter_sent{base_ctrl}', data=long).fit(cov_type='HC3')
m3 = smf.ols(f'pct_return ~ news_sent{base_ctrl}', data=long).fit(cov_type='HC3')
m4 = smf.ols(f'pct_return ~ twitter_sent + news_sent{base_ctrl}', data=long).fit(cov_type='HC3')

print('SET 1 — Model 1 (twitter_sent):')
print(f"  twitter_sent: {m1.params['twitter_sent']:.6f}  ({m1.bse['twitter_sent']:.6f}) {stars(m1.pvalues['twitter_sent'])}")
print(f"  N={int(m1.nobs)}, R²={m1.rsquared:.4f}")

print('\nSET 1 — Model 3 (news_sent):')
print(f"  news_sent: {m3.params['news_sent']:.6f}  ({m3.bse['news_sent']:.6f}) {stars(m3.pvalues['news_sent'])}")
print(f"  N={int(m3.nobs)}, R²={m3.rsquared:.4f}")

print('\nSET 1 — Model 4 (twitter_sent + news_sent):')
print(f"  twitter_sent: {m4.params['twitter_sent']:.6f}  ({m4.bse['twitter_sent']:.6f}) {stars(m4.pvalues['twitter_sent'])}")
print(f"  news_sent:    {m4.params['news_sent']:.6f}  ({m4.bse['news_sent']:.6f}) {stars(m4.pvalues['news_sent'])}")
print(f"  N={int(m4.nobs)}, R²={m4.rsquared:.4f}")

In [ ]:
# Model 2: polarity-broken counts (requires twitter_pos_count and twitter_neu_count)
if HAS_POS_NEU:
    sub2 = long.dropna(subset=['twitter_pos_count', 'twitter_neu_count', 'twitter_neg_count'])
    m2 = smf.ols('pct_return ~ twitter_pos_count + twitter_neu_count + twitter_neg_count', data=sub2).fit(cov_type='HC3')
    print('SET 1 — Model 2 (polarity-broken tweet counts):')
    for p in ['twitter_pos_count', 'twitter_neu_count', 'twitter_neg_count']:
        print(f'  {p}: {m2.params[p]:.6e}  ({m2.bse[p]:.6e}) {stars(m2.pvalues[p])}')
    print(f'  N={int(m2.nobs)}')
    print('  Teti et al. finding: pos significant+, neu not significant, neg significant−')
else:
    m2 = None
    print('[SKIPPED] Model 2: pull twitter_pos_count + twitter_neu_count from Bloomberg')

## 5. SET 2 — Panel FE with Binary Treatment
Replicates Table 4 of Teti et al. For each lag k ∈ {0,1,2,3,4}:

$$\text{pct\_return}_{ct} = \beta_0 + \beta_1 \cdot \text{ts\_lag}_k + \beta_2 \cdot \text{group} + \beta_3 \cdot (\text{ts\_lag}_k \times \text{group}) + \beta_4 \cdot \text{volume} + \text{EntityFE} + \varepsilon$$

**Key parameter:** β₃ — incremental sentiment effect for high social-media coverage firms.  
Teti et al. find the 3-day lag significant with Bloomberg composite; 1-day lag with ts_b.

In [ ]:
panel = long.copy().set_index(['ticker', 'date'])

print(f'SET 2 — Panel FE with Binary Treatment ({group_note})')
print('Key coefficient: β₃ = ts_lag_k × group_dummy\n')

set2_results = {}
for k in range(5):
    lag_col   = f'ts_lag_{k}'
    inter_col = f'inter_{k}'
    panel[inter_col] = panel[lag_col] * panel['group']
    sub = panel[[lag_col, inter_col, 'group', 'volume_mil', 'pct_return']].dropna()
    if len(sub) < 50:
        print(f'  Lag {k}: insufficient data')
        continue
    try:
        res = PanelOLS.from_formula(
            f'pct_return ~ {lag_col} + group + {inter_col} + volume_mil + EntityEffects',
            data=sub
        ).fit(cov_type='heteroskedastic')
        set2_results[k] = res
        c = res.params[inter_col]
        s = res.std_errors[inter_col]
        p = res.pvalues[inter_col]
        print(f'  Lag {k}: β₃ = {c:.6f}  ({s:.6f}) {stars(p)}  [N={res.nobs}]')
    except Exception as e:
        print(f'  Lag {k} failed: {e}')

## 6. SET 3 — Polarity-Weighted Index (ts_b)
Replicates Table 5 of Teti et al. Replaces Bloomberg composite with ts_b, which assigns zero weight to neutral tweets in the numerator — eliminating the dilution effect that weakens the Bloomberg index.

Teti et al. finding: **1-day lag is significant** with ts_b, vs 3-day with Bloomberg composite. The faster signal suggests neutral-tweet dilution delays the detected effect.

In [ ]:
print(f'SET 3 — Custom Polarity-Weighted Index ({ts_b_note})')
print('Key coefficient: β₃ = tsb_lag_k × group_dummy\n')

set3_results = {}
for k in range(5):
    lag_col   = f'tsb_lag_{k}'
    inter_col = f'interb_{k}'
    panel[inter_col] = panel[lag_col] * panel['group']
    sub = panel[[lag_col, inter_col, 'group', 'volume_mil', 'pct_return']].dropna()
    if len(sub) < 50:
        print(f'  Lag {k}: insufficient data')
        continue
    try:
        res = PanelOLS.from_formula(
            f'pct_return ~ {lag_col} + group + {inter_col} + volume_mil + EntityEffects',
            data=sub
        ).fit(cov_type='heteroskedastic')
        set3_results[k] = res
        c = res.params[inter_col]
        s = res.std_errors[inter_col]
        p = res.pvalues[inter_col]
        print(f'  Lag {k}: β₃ = {c:.6f}  ({s:.6f}) {stars(p)}  [N={res.nobs}]')
    except Exception as e:
        print(f'  Lag {k} failed: {e}')

## 7. Export — Publication-ready LaTeX tables

In [ ]:
def build_latex_table(rows, caption, label, notes=''):
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        f'\\caption{{{caption}}}', f'\\label{{{label}}}',
        r'\begin{tabular}{lcc}', r'\hline\hline',
        r'Variable & Coefficient & (SE) \\', r'\hline',
    ]
    for r in rows:
        s = stars(r['pval'])
        lines.append(f"{r['label']} & {r['coef']:.6f}{s} & ({r['se']:.6f}) \\\\")
    if notes:
        lines.append(r'\hline\hline')
        lines.append(f'\\multicolumn{{3}}{{l}}{{\\footnotesize \\textit{{Notes:}} {notes}}} \\\\')
    lines += [r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

# SET 1 table
set1_rows = [
    {'label': r'twitter\_sent (M1)', 'coef': m1.params['twitter_sent'],
     'se': m1.bse['twitter_sent'], 'pval': m1.pvalues['twitter_sent']},
    {'label': r'news\_sent (M3)', 'coef': m3.params['news_sent'],
     'se': m3.bse['news_sent'], 'pval': m3.pvalues['news_sent']},
    {'label': r'twitter\_sent (M4)', 'coef': m4.params['twitter_sent'],
     'se': m4.bse['twitter_sent'], 'pval': m4.pvalues['twitter_sent']},
    {'label': r'news\_sent (M4)', 'coef': m4.params['news_sent'],
     'se': m4.bse['news_sent'], 'pval': m4.pvalues['news_sent']},
]
tex_set1 = build_latex_table(set1_rows,
    caption='Teti et al. (2019) Replication --- SET 1: Pooled OLS',
    label='tab:teti_set1',
    notes='HC3-robust SEs. Dependent variable: pct open-to-close return. *** p$<$0.01, ** p$<$0.05, * p$<$0.1.')

# SET 2 table
set2_rows = []
for k, res in set2_results.items():
    ic = f'inter_{k}'
    if ic in res.params:
        set2_rows.append({'label': f'ts\\_lag\\_{k} $\\times$ group',
                          'coef': float(res.params[ic]),
                          'se': float(res.std_errors[ic]),
                          'pval': float(res.pvalues[ic])})
tex_set2 = build_latex_table(set2_rows,
    caption='Teti et al. (2019) Replication --- SET 2: FE Binary Treatment ($\\beta_3$ only)',
    label='tab:teti_set2',
    notes=f'Heteroskedasticity-robust SEs. Entity FE. {group_note}. *** p$<$0.01, ** p$<$0.05, * p$<$0.1.')

# SET 3 table
set3_rows = []
for k, res in set3_results.items():
    ic = f'interb_{k}'
    if ic in res.params:
        set3_rows.append({'label': f'tsb\\_lag\\_{k} $\\times$ group',
                          'coef': float(res.params[ic]),
                          'se': float(res.std_errors[ic]),
                          'pval': float(res.pvalues[ic])})
tex_set3 = build_latex_table(set3_rows,
    caption='Teti et al. (2019) Replication --- SET 3: Polarity-Weighted Index (ts\\_b)',
    label='tab:teti_set3',
    notes=f'{ts_b_note}. Heteroskedasticity-robust SEs. Entity FE. *** p$<$0.01, ** p$<$0.05, * p$<$0.1.')

for fname, content in [('teti_set1.tex', tex_set1), ('teti_set2.tex', tex_set2), ('teti_set3.tex', tex_set3)]:
    path = OUTPUT_PATH + fname
    with open(path, 'w') as f:
        f.write(content)
    print(f'Saved {path}')

print(tex_set1)

In [ ]:
# Push outputs to GitHub
import os, subprocess
os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name', 'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])
staged = subprocess.run(['git', 'diff', '--cached', '--quiet'])
unpushed = subprocess.run(['git', 'log', 'origin/main..HEAD', '--oneline'], capture_output=True, text=True)
if staged.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add Teti et al. replication results'], check=True)
if staged.returncode != 0 or unpushed.stdout.strip():
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit.')